# Part 2 Verification — ETL Pipeline Step by Step

Run Part 1 first (Part1_Group11.sql), then execute each ETL step here and observe how data transforms:

1. **Source** (WideWorldImporters) → 3NF, scattered across tables
2. **Stage** (after Extract) → flat, raw copy from source
3. **PreLoad** (after Transform) → surrogate keys + SCD logic applied
4. **Dim/Fact** (after Load) → final Star Schema

In [ ]:
import pyodbc
import pandas as pd
import re

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=localhost;"
    r"DATABASE=WWI_DM;"
    r"Trusted_Connection=yes;"
)

def sql(query):
    return pd.read_sql(query, conn)

def execute(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    batches = re.split(r'(?m)^\s*GO\s*$', content)
    cursor = conn.cursor()
    executed = 0
    for i, batch in enumerate(batches):
        lines = [l for l in batch.strip().splitlines() if l.strip() and not l.strip().startswith('--')]
        if not lines:
            continue
        try:
            cursor.execute(batch.strip())
            conn.commit()
            executed += 1
            print(f"  Batch {i+1}: OK")
        except Exception as e:
            print(f"  Batch {i+1}: ERROR - {e}")
    print(f"Done: {filepath} ({executed} batches executed)")

def run(sql_text):
    cursor = conn.cursor()
    cursor.execute(sql_text)
    conn.commit()

print("Connected to WWI_DM")

## Step 0: Create infrastructure (Stage tables + PreLoad tables + Sequences + all SPs)

In [ ]:
execute("../../part2/req4-extract/create-staging-tables.sql")
execute("../../part2/req4-extract/t-sql/extract-sps.sql")
execute("../../part2/req5-transform/create-preload-tables.sql")
execute("../../part2/req5-transform/t-sql/transform-sps.sql")
execute("../../part2/req6-load/t-sql/load-sps.sql")

## Step 1: Extract (Req 4) — Source → Stage

Pulls data from WideWorldImporters (3NF) into flat Stage tables. Business keys only, no surrogate keys.

In [ ]:
run("EXEC dbo.Customers_Extract")
run("EXEC dbo.Products_Extract")
run("EXEC dbo.SalesPeople_Extract")
run("EXEC dbo.Suppliers_Extract")
run("EXEC dbo.Orders_Extract @OrderDate = '2013-01-01'")
print("Extract complete for 2013-01-01")

In [ ]:
print("=== Customers_Stage (top 3) — names only, no keys ===")
display(sql("SELECT TOP 3 * FROM dbo.Customers_Stage"))

print("\n=== Orders_Stage (top 3) — business keys (CustomerName, StockItemName, LogonName) ===")
display(sql("SELECT TOP 3 * FROM dbo.Orders_Stage"))

print("\n=== Row counts ===")
display(sql("""
    SELECT 'Customers_Stage' AS T, COUNT(*) AS Rows FROM dbo.Customers_Stage
    UNION ALL SELECT 'Products_Stage', COUNT(*) FROM dbo.Products_Stage
    UNION ALL SELECT 'SalesPeople_Stage', COUNT(*) FROM dbo.SalesPeople_Stage
    UNION ALL SELECT 'Suppliers_Stage', COUNT(*) FROM dbo.Suppliers_Stage
    UNION ALL SELECT 'Orders_Stage', COUNT(*) FROM dbo.Orders_Stage
"""))

## Step 2: Transform (Req 5) — Stage → PreLoad

Maps business keys → surrogate keys, handles SCD logic, aggregates measures.

In [ ]:
run("EXEC dbo.Location_Transform")
run("EXEC dbo.Customers_Transform")
run("EXEC dbo.Products_Transform")
run("EXEC dbo.SalesPeople_Transform")
run("EXEC dbo.Suppliers_Transform")
run("EXEC dbo.Orders_Transform")
print("Transform complete")

### Compare: Stage vs PreLoad — what changed?

In [ ]:
print("=== Customers_Stage (first row) — business key only ===")
display(sql("SELECT TOP 1 * FROM dbo.Customers_Stage"))

print("\n=== Customers_Preload (first row) — surrogate key + SCD dates added ===")
display(sql("SELECT TOP 1 * FROM dbo.Customers_Preload ORDER BY CustomerKey"))

print("\n=== Orders_Stage (first row) — business keys (names) ===")
display(sql("SELECT TOP 1 * FROM dbo.Orders_Stage"))

print("\n=== Orders_Preload (first row) — surrogate keys (integers) + calculated measures ===")
display(sql("SELECT TOP 1 * FROM dbo.Orders_Preload"))

## Step 3: Load (Req 6) — PreLoad → Dim/Fact

Dims first (DELETE+INSERT), then Fact (INSERT only).

In [ ]:
run("EXEC dbo.Location_Load")
run("EXEC dbo.Customers_Load")
run("EXEC dbo.Products_Load")
run("EXEC dbo.SalesPeople_Load")
run("EXEC dbo.Suppliers_Load")
run("EXEC dbo.Orders_Load")
print("Load complete — Day 1 (2013-01-01)")

In [ ]:
print("=== Day 1 row counts ===")
display(sql("""
    SELECT 'DimLocation' AS T, COUNT(*) AS Rows FROM dbo.DimLocation
    UNION ALL SELECT 'DimCustomers', COUNT(*) FROM dbo.DimCustomers
    UNION ALL SELECT 'DimProducts', COUNT(*) FROM dbo.DimProducts
    UNION ALL SELECT 'DimSalesPeople', COUNT(*) FROM dbo.DimSalesPeople
    UNION ALL SELECT 'DimSuppliers', COUNT(*) FROM dbo.DimSuppliers
    UNION ALL SELECT 'FactSales', COUNT(*) FROM dbo.FactSales
"""))

## Step 4: Run remaining 3 days (Req 7)

In [ ]:
from datetime import date, timedelta

for day_offset in range(1, 4):
    d = date(2013, 1, 1) + timedelta(days=day_offset)
    print(f"===== Processing: {d} =====")
    run("EXEC dbo.Customers_Extract")
    run("EXEC dbo.Products_Extract")
    run("EXEC dbo.SalesPeople_Extract")
    run("EXEC dbo.Suppliers_Extract")
    run(f"EXEC dbo.Orders_Extract @OrderDate = '{d}'")
    run("EXEC dbo.Location_Transform")
    run("EXEC dbo.Customers_Transform")
    run("EXEC dbo.Products_Transform")
    run("EXEC dbo.SalesPeople_Transform")
    run("EXEC dbo.Suppliers_Transform")
    run("EXEC dbo.Orders_Transform")
    run("EXEC dbo.Location_Load")
    run("EXEC dbo.Customers_Load")
    run("EXEC dbo.Products_Load")
    run("EXEC dbo.SalesPeople_Load")
    run("EXEC dbo.Suppliers_Load")
    run("EXEC dbo.Orders_Load")
    fact_count = sql("SELECT COUNT(*) AS Rows FROM dbo.FactSales").iloc[0]['Rows']
    print(f"  FactSales total: {fact_count} rows\n")

print("All 4 days complete.")

## Final Verification

In [ ]:
print("=== Final row counts (4 days of ETL) ===")
display(sql("""
    SELECT 'DimLocation' AS T, COUNT(*) AS Rows FROM dbo.DimLocation
    UNION ALL SELECT 'DimCustomers', COUNT(*) FROM dbo.DimCustomers
    UNION ALL SELECT 'DimProducts', COUNT(*) FROM dbo.DimProducts
    UNION ALL SELECT 'DimSalesPeople', COUNT(*) FROM dbo.DimSalesPeople
    UNION ALL SELECT 'DimSuppliers', COUNT(*) FROM dbo.DimSuppliers
    UNION ALL SELECT 'DimDate', COUNT(*) FROM dbo.DimDate
    UNION ALL SELECT 'FactSales', COUNT(*) FROM dbo.FactSales
"""))

✅ **Part 2 passed if:**
- FactSales has 667 rows (4 days of orders)
- All Dim tables populated
- No SQL errors throughout the pipeline